# 05 Recent Form Features

This notebook adds recent-performance features to the NFL forecasting model.

The previous model used full-season pre-game averages. This notebook adds rolling last-3-game features so the model can better capture how teams have been performing recently.

In [1]:
# Import packages
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# Load game results
games = pd.read_csv("../data/processed/game_results_2023_2025.csv")

games.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0


In [3]:
# Sort games
games["gameday"] = pd.to_datetime(games["gameday"])

games = games.sort_values(["season", "week", "gameday"]).reset_index(drop=True)

games.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0


In [4]:
# Create home team rows
home_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_rows = home_rows.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_scored",
        "away_score": "points_allowed"
    }
)

home_rows["is_home"] = 1

In [5]:
# Create away team rows
away_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_rows = away_rows.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_scored",
        "home_score": "points_allowed"
    }
)

away_rows["is_home"] = 0

In [6]:
# Combine team-game rows
team_games = pd.concat([home_rows, away_rows], ignore_index=True)

team_games = team_games.sort_values(["team", "season", "week", "gameday"]).reset_index(drop=True)

team_games.head(10)

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home
0,2023,1,2023_01_ARI_WAS,2023-09-10,ARI,WAS,16.0,20.0,0
1,2023,2,2023_02_NYG_ARI,2023-09-17,ARI,NYG,28.0,31.0,1
2,2023,3,2023_03_DAL_ARI,2023-09-24,ARI,DAL,28.0,16.0,1
3,2023,4,2023_04_ARI_SF,2023-10-01,ARI,SF,16.0,35.0,0
4,2023,5,2023_05_CIN_ARI,2023-10-08,ARI,CIN,20.0,34.0,1
5,2023,6,2023_06_ARI_LA,2023-10-15,ARI,LA,9.0,26.0,0
6,2023,7,2023_07_ARI_SEA,2023-10-22,ARI,SEA,10.0,20.0,0
7,2023,8,2023_08_BAL_ARI,2023-10-29,ARI,BAL,24.0,31.0,1
8,2023,9,2023_09_ARI_CLE,2023-11-05,ARI,CLE,0.0,27.0,0
9,2023,10,2023_10_ATL_ARI,2023-11-12,ARI,ATL,25.0,23.0,1


In [7]:
# Add result columns
team_games["point_diff"] = team_games["points_scored"] - team_games["points_allowed"]

team_games["team_won"] = (team_games["points_scored"] > team_games["points_allowed"]).astype(int)

team_games.head()

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home,point_diff,team_won
0,2023,1,2023_01_ARI_WAS,2023-09-10,ARI,WAS,16.0,20.0,0,-4.0,0
1,2023,2,2023_02_NYG_ARI,2023-09-17,ARI,NYG,28.0,31.0,1,-3.0,0
2,2023,3,2023_03_DAL_ARI,2023-09-24,ARI,DAL,28.0,16.0,1,12.0,1
3,2023,4,2023_04_ARI_SF,2023-10-01,ARI,SF,16.0,35.0,0,-19.0,0
4,2023,5,2023_05_CIN_ARI,2023-10-08,ARI,CIN,20.0,34.0,1,-14.0,0


In [8]:
# Add season-long pre-game features
team_games["games_played_before"] = team_games.groupby(["team", "season"]).cumcount()

team_games["points_scored_before"] = (
    team_games.groupby(["team", "season"])["points_scored"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["points_allowed_before"] = (
    team_games.groupby(["team", "season"])["points_allowed"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["point_diff_before"] = (
    team_games.groupby(["team", "season"])["point_diff"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["wins_before"] = (
    team_games.groupby(["team", "season"])["team_won"]
    .transform(lambda x: x.cumsum().shift(1))
)

cols_to_fill = [
    "points_scored_before",
    "points_allowed_before",
    "point_diff_before",
    "wins_before"
]

team_games[cols_to_fill] = team_games[cols_to_fill].fillna(0)

In [9]:
# Create season-long averages
team_games["avg_points_scored_before"] = 0.0
team_games["avg_points_allowed_before"] = 0.0
team_games["avg_point_diff_before"] = 0.0
team_games["win_pct_before"] = 0.0

mask = team_games["games_played_before"] > 0

team_games.loc[mask, "avg_points_scored_before"] = (
    team_games.loc[mask, "points_scored_before"] / team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "avg_points_allowed_before"] = (
    team_games.loc[mask, "points_allowed_before"] / team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "avg_point_diff_before"] = (
    team_games.loc[mask, "point_diff_before"] / team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "win_pct_before"] = (
    team_games.loc[mask, "wins_before"] / team_games.loc[mask, "games_played_before"]
)

In [10]:
# Create last-3-game rolling features
team_games["last3_avg_points_scored"] = (
    team_games.groupby(["team", "season"])["points_scored"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

team_games["last3_avg_points_allowed"] = (
    team_games.groupby(["team", "season"])["points_allowed"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

team_games["last3_avg_point_diff"] = (
    team_games.groupby(["team", "season"])["point_diff"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

team_games["last3_win_pct"] = (
    team_games.groupby(["team", "season"])["team_won"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

In [11]:
# Fill missing last-3 values
last3_cols = [
    "last3_avg_points_scored",
    "last3_avg_points_allowed",
    "last3_avg_point_diff",
    "last3_win_pct"
]

team_games[last3_cols] = team_games[last3_cols].fillna(0)

In [12]:
# Check one team
team_games[team_games["team"] == "KC"][
    [
        "season",
        "week",
        "team",
        "opponent",
        "points_scored",
        "points_allowed",
        "games_played_before",
        "avg_points_scored_before",
        "last3_avg_points_scored",
        "avg_points_allowed_before",
        "last3_avg_points_allowed",
        "avg_point_diff_before",
        "last3_avg_point_diff",
        "win_pct_before",
        "last3_win_pct"
    ]
].head(15)

,season,week,team,opponent,points_scored,points_allowed,games_played_before,avg_points_scored_before,last3_avg_points_scored,avg_points_allowed_before,last3_avg_points_allowed,avg_point_diff_before,last3_avg_point_diff,win_pct_before,last3_win_pct
799,2023,1,KC,DET,20.0,21.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
800,2023,2,KC,JAX,17.0,9.0,1,20.000000,20.000000,21.000000,21.000000,-1.000000,-1.000000,0.000000,0.000000
801,2023,3,KC,CHI,41.0,10.0,2,18.500000,18.500000,15.000000,15.000000,3.500000,3.500000,0.500000,0.500000
802,2023,4,KC,NYJ,23.0,20.0,3,26.000000,26.000000,13.333333,13.333333,12.666667,12.666667,0.666667,0.666667
803,2023,5,KC,MIN,27.0,20.0,4,25.250000,27.000000,15.000000,13.000000,10.250000,14.000000,0.750000,1.000000
804,2023,6,KC,DEN,19.0,8.0,5,25.600000,30.333333,16.000000,16.666667,9.600000,13.666667,0.800000,1.000000
805,2023,7,KC,LAC,31.0,17.0,6,24.500000,23.000000,14.666667,16.000000,9.833333,7.000000,0.833333,1.000000
806,2023,8,KC,DEN,9.0,24.0,7,25.428571,25.666667,15.000000,15.000000,10.428571,10.666667,0.857143,1.000000
807,2023,9,KC,MIA,21.0,14.0,8,23.375000,19.666667,16.125000,16.333333,7.250000,3.333333,0.750000,0.666667
808,2023,11,KC,PHI,17.0,21.0,9,23.111111,20.333333,15.888889,18.333333,7.222222,2.000000,0.777778,0.666667


In [13]:
# Create home features
home_features = team_games[team_games["is_home"] == 1][
    [
        "game_id",
        "team",
        "games_played_before",
        "avg_points_scored_before",
        "avg_points_allowed_before",
        "avg_point_diff_before",
        "win_pct_before",
        "last3_avg_points_scored",
        "last3_avg_points_allowed",
        "last3_avg_point_diff",
        "last3_win_pct"
    ]
].copy()

home_features = home_features.rename(
    columns={
        "team": "home_team_check",
        "games_played_before": "home_games_played_before",
        "avg_points_scored_before": "home_avg_points_scored_before",
        "avg_points_allowed_before": "home_avg_points_allowed_before",
        "avg_point_diff_before": "home_avg_point_diff_before",
        "win_pct_before": "home_win_pct_before",
        "last3_avg_points_scored": "home_last3_avg_points_scored",
        "last3_avg_points_allowed": "home_last3_avg_points_allowed",
        "last3_avg_point_diff": "home_last3_avg_point_diff",
        "last3_win_pct": "home_last3_win_pct"
    }
)

home_features.head()

,game_id,home_team_check,home_games_played_before,home_avg_points_scored_before,home_avg_points_allowed_before,home_avg_point_diff_before,home_win_pct_before,home_last3_avg_points_scored,home_last3_avg_points_allowed,home_last3_avg_point_diff,home_last3_win_pct
1,2023_02_NYG_ARI,ARI,1,16.000000,20.000000,-4.000000,0.000000,16.000000,20.000000,-4.000000,0.000000
2,2023_03_DAL_ARI,ARI,2,22.000000,25.500000,-3.500000,0.000000,22.000000,25.500000,-3.500000,0.000000
4,2023_05_CIN_ARI,ARI,4,22.000000,25.500000,-3.500000,0.250000,24.000000,27.333333,-3.333333,0.333333
7,2023_08_BAL_ARI,ARI,7,18.142857,26.000000,-7.857143,0.142857,13.000000,26.666667,-13.666667,0.000000
9,2023_10_ATL_ARI,ARI,9,16.777778,26.666667,-9.888889,0.111111,11.333333,26.000000,-14.666667,0.000000


In [14]:
# Create away features
away_features = team_games[team_games["is_home"] == 0][
    [
        "game_id",
        "team",
        "games_played_before",
        "avg_points_scored_before",
        "avg_points_allowed_before",
        "avg_point_diff_before",
        "win_pct_before",
        "last3_avg_points_scored",
        "last3_avg_points_allowed",
        "last3_avg_point_diff",
        "last3_win_pct"
    ]
].copy()

away_features = away_features.rename(
    columns={
        "team": "away_team_check",
        "games_played_before": "away_games_played_before",
        "avg_points_scored_before": "away_avg_points_scored_before",
        "avg_points_allowed_before": "away_avg_points_allowed_before",
        "avg_point_diff_before": "away_avg_point_diff_before",
        "win_pct_before": "away_win_pct_before",
        "last3_avg_points_scored": "away_last3_avg_points_scored",
        "last3_avg_points_allowed": "away_last3_avg_points_allowed",
        "last3_avg_point_diff": "away_last3_avg_point_diff",
        "last3_win_pct": "away_last3_win_pct"
    }
)

away_features.head()

,game_id,away_team_check,away_games_played_before,away_avg_points_scored_before,away_avg_points_allowed_before,away_avg_point_diff_before,away_win_pct_before,away_last3_avg_points_scored,away_last3_avg_points_allowed,away_last3_avg_point_diff,away_last3_win_pct
0,2023_01_ARI_WAS,ARI,0,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2023_04_ARI_SF,ARI,3,24.000,22.333333,1.666667,0.333333,24.000000,22.333333,1.666667,0.333333
5,2023_06_ARI_LA,ARI,5,21.600,27.200000,-5.600000,0.200000,21.333333,28.333333,-7.000000,0.333333
6,2023_07_ARI_SEA,ARI,6,19.500,27.000000,-7.500000,0.166667,15.000000,31.666667,-16.666667,0.000000
8,2023_09_ARI_CLE,ARI,8,18.875,26.625000,-7.750000,0.125000,14.333333,25.666667,-11.333333,0.000000


In [15]:
# Merge features onto games
modeling_data_recent = games.merge(home_features, on="game_id", how="left")

modeling_data_recent = modeling_data_recent.merge(away_features, on="game_id", how="left")

modeling_data_recent.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,away_team_check,away_games_played_before,away_avg_points_scored_before,away_avg_points_allowed_before,away_avg_point_diff_before,away_win_pct_before,away_last3_avg_points_scored,away_last3_avg_points_allowed,away_last3_avg_point_diff,away_last3_win_pct
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0,...,DET,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0,...,CAR,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0,...,HOU,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0,...,CIN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0,...,JAX,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# Check team matches
modeling_data_recent[
    [
        "game_id",
        "home_team",
        "home_team_check",
        "away_team",
        "away_team_check"
    ]
].head(10)

,game_id,home_team,home_team_check,away_team,away_team_check
0,2023_01_DET_KC,KC,KC,DET,DET
1,2023_01_CAR_ATL,ATL,ATL,CAR,CAR
2,2023_01_HOU_BAL,BAL,BAL,HOU,HOU
3,2023_01_CIN_CLE,CLE,CLE,CIN,CIN
4,2023_01_JAX_IND,IND,IND,JAX,JAX
5,2023_01_TB_MIN,MIN,MIN,TB,TB
6,2023_01_TEN_NO,NO,NO,TEN,TEN
7,2023_01_SF_PIT,PIT,PIT,SF,SF
8,2023_01_ARI_WAS,WAS,WAS,ARI,ARI
9,2023_01_GB_CHI,CHI,CHI,GB,GB


In [17]:
# Drop check columns
modeling_data_recent = modeling_data_recent.drop(columns=["home_team_check", "away_team_check"])

In [18]:
# Create difference features
modeling_data_recent["avg_points_scored_diff"] = (
    modeling_data_recent["home_avg_points_scored_before"] - modeling_data_recent["away_avg_points_scored_before"]
)

modeling_data_recent["avg_points_allowed_diff"] = (
    modeling_data_recent["home_avg_points_allowed_before"] - modeling_data_recent["away_avg_points_allowed_before"]
)

modeling_data_recent["avg_point_diff_diff"] = (
    modeling_data_recent["home_avg_point_diff_before"] - modeling_data_recent["away_avg_point_diff_before"]
)

modeling_data_recent["win_pct_diff"] = (
    modeling_data_recent["home_win_pct_before"] - modeling_data_recent["away_win_pct_before"]
)

modeling_data_recent["last3_avg_points_scored_diff"] = (
    modeling_data_recent["home_last3_avg_points_scored"] - modeling_data_recent["away_last3_avg_points_scored"]
)

modeling_data_recent["last3_avg_points_allowed_diff"] = (
    modeling_data_recent["home_last3_avg_points_allowed"] - modeling_data_recent["away_last3_avg_points_allowed"]
)

modeling_data_recent["last3_avg_point_diff_diff"] = (
    modeling_data_recent["home_last3_avg_point_diff"] - modeling_data_recent["away_last3_avg_point_diff"]
)

modeling_data_recent["last3_win_pct_diff"] = (
    modeling_data_recent["home_last3_win_pct"] - modeling_data_recent["away_last3_win_pct"]
)

In [19]:
# Preview new features
modeling_data_recent[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_team_won",
        "avg_point_diff_diff",
        "last3_avg_point_diff_diff",
        "win_pct_diff",
        "last3_win_pct_diff"
    ]
].head(15)

,season,week,home_team,away_team,home_team_won,avg_point_diff_diff,last3_avg_point_diff_diff,win_pct_diff,last3_win_pct_diff
0,2023,1,KC,DET,0,0.0,0.0,0.0,0.0
1,2023,1,ATL,CAR,1,0.0,0.0,0.0,0.0
2,2023,1,BAL,HOU,1,0.0,0.0,0.0,0.0
3,2023,1,CLE,CIN,1,0.0,0.0,0.0,0.0
4,2023,1,IND,JAX,0,0.0,0.0,0.0,0.0
5,2023,1,MIN,TB,0,0.0,0.0,0.0,0.0
6,2023,1,NO,TEN,1,0.0,0.0,0.0,0.0
7,2023,1,PIT,SF,0,0.0,0.0,0.0,0.0
8,2023,1,WAS,ARI,1,0.0,0.0,0.0,0.0
9,2023,1,CHI,GB,0,0.0,0.0,0.0,0.0


In [20]:
# Save recent-form modeling dataset
modeling_data_recent.to_csv("../data/processed/modeling_dataset_recent_form_2023_2025.csv", index=False)

print("Saved recent-form modeling dataset.")
print(modeling_data_recent.shape)

Saved recent-form modeling dataset.
(855, 36)


In [21]:
# Choose model features
features = [
    "avg_points_scored_diff",
    "avg_points_allowed_diff",
    "avg_point_diff_diff",
    "win_pct_diff",
    "last3_avg_points_scored_diff",
    "last3_avg_points_allowed_diff",
    "last3_avg_point_diff_diff",
    "last3_win_pct_diff"
]

target = "home_team_won"

In [22]:
# Train on 2023–2024 and test on 2025
train_data = modeling_data_recent[modeling_data_recent["season"].isin([2023, 2024])].copy()
test_data = modeling_data_recent[modeling_data_recent["season"] == 2025].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 570
Testing rows: 285


In [23]:
# Train model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [24]:
# Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [25]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Recent-form model accuracy: {accuracy:.2%}")

Recent-form model accuracy: 61.40%


In [26]:
# Home-team baseline
home_team_baseline = [1] * len(y_test)

baseline_accuracy = accuracy_score(y_test, home_team_baseline)

print(f"Baseline accuracy if always picking home team: {baseline_accuracy:.2%}")

Baseline accuracy if always picking home team: 53.33%


In [27]:
# Classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.43      0.51       133
           1       0.61      0.78      0.68       152

    accuracy                           0.61       285
   macro avg       0.62      0.60      0.60       285
weighted avg       0.62      0.61      0.60       285



In [28]:
# Coefficients
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

coefficients.sort_values("coefficient", ascending=False)

,feature,coefficient
3,win_pct_diff,0.371221
7,last3_win_pct_diff,0.051541
0,avg_points_scored_diff,0.027480
2,avg_point_diff_diff,0.019971
1,avg_points_allowed_diff,0.007510
6,last3_avg_point_diff_diff,0.004026
4,last3_avg_points_scored_diff,0.002077
5,last3_avg_points_allowed_diff,-0.001949


In [29]:
# Results table
results = test_data[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won"
    ]
].copy()

results["predicted_home_win"] = y_pred
results["home_win_probability"] = y_prob

results["predicted_winner"] = results.apply(
    lambda row: row["home_team"] if row["predicted_home_win"] == 1 else row["away_team"],
    axis=1
)

results["actual_winner"] = results.apply(
    lambda row: row["home_team"] if row["home_team_won"] == 1 else row["away_team"],
    axis=1
)

results["correct_prediction"] = (
    results["predicted_winner"] == results["actual_winner"]
)

results.head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,predicted_home_win,home_win_probability,predicted_winner,actual_winner,correct_prediction
570,2025,1,2025_01_DAL_PHI,2025-09-04,PHI,DAL,24.0,20.0,1,1,0.558613,PHI,PHI,True
571,2025,1,2025_01_KC_LAC,2025-09-05,LAC,KC,27.0,21.0,1,1,0.558613,LAC,LAC,True
572,2025,1,2025_01_TB_ATL,2025-09-07,ATL,TB,20.0,23.0,0,1,0.558613,ATL,TB,False
573,2025,1,2025_01_CIN_CLE,2025-09-07,CLE,CIN,16.0,17.0,0,1,0.558613,CLE,CIN,False
574,2025,1,2025_01_MIA_IND,2025-09-07,IND,MIA,33.0,8.0,1,1,0.558613,IND,IND,True
575,2025,1,2025_01_CAR_JAX,2025-09-07,JAX,CAR,26.0,10.0,1,1,0.558613,JAX,JAX,True
576,2025,1,2025_01_LV_NE,2025-09-07,NE,LV,13.0,20.0,0,1,0.558613,NE,LV,False
577,2025,1,2025_01_ARI_NO,2025-09-07,NO,ARI,13.0,20.0,0,1,0.558613,NO,ARI,False
578,2025,1,2025_01_PIT_NYJ,2025-09-07,NYJ,PIT,32.0,34.0,0,1,0.558613,NYJ,PIT,False
579,2025,1,2025_01_NYG_WAS,2025-09-07,WAS,NYG,21.0,6.0,1,1,0.558613,WAS,WAS,True


In [30]:
# Accuracy by week
accuracy_by_week = results.groupby("week")["correct_prediction"].mean().reset_index()

accuracy_by_week["accuracy_percent"] = accuracy_by_week["correct_prediction"] * 100

accuracy_by_week = accuracy_by_week.drop(columns=["correct_prediction"])

accuracy_by_week

,week,accuracy_percent
0,1,56.250000
1,2,68.750000
2,3,56.250000
3,4,50.000000
4,5,42.857143
5,6,53.333333
6,7,73.333333
7,8,69.230769
8,9,57.142857
9,10,64.285714


In [31]:
# Save recent-form predictions
results.to_csv("../data/predictions/recent_form_model_test_predictions.csv", index=False)

print("Saved recent-form model test predictions.")
print(results.shape)

Saved recent-form model test predictions.
(285, 14)


In [32]:
# Final summary numbers
print("Recent-form model accuracy:", f"{accuracy:.2%}")
print("Previous season-based baseline: 61.40%")
print("Home-team baseline accuracy:", f"{baseline_accuracy:.2%}")
print("Number of test games:", len(results))
print("Correct predictions:", results["correct_prediction"].sum())
print("Incorrect predictions:", len(results) - results["correct_prediction"].sum())

Recent-form model accuracy: 61.40%
Previous season-based baseline: 61.40%
Home-team baseline accuracy: 53.33%
Number of test games: 285
Correct predictions: 175
Incorrect predictions: 110


## Summary

In this notebook, I added recent-form features to the NFL forecasting model.

The new features used each team's last 3 games before the matchup. This helped the model consider more recent team performance instead of only relying on season-long averages.

The next step is to compare this model's accuracy to the previous season-based baseline of 61.40%. If the recent-form model improves accuracy, these features should be included in future versions of the forecasting model.